<a href="https://colab.research.google.com/github/parinyad123/financial-analyst-agent/blob/main/notebooks/financial_agent_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-google-genai langgraph yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.66
    Uninstalling yfinance-0.2.66:
      Successfully uninstalled yfinance-0.2.66


In [2]:
!pip install -q langsmith --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.2/481.2 kB 2.6 MB/s eta 0:00:00


In [9]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.2 MB/s eta 0:00:00


In [3]:
from langsmith import Client
ls_client = Client()
print("✅ OK")

✅ OK


In [4]:
# เช็ค version
import langsmith
print(langsmith.__version__)

0.8.14


In [5]:
import os
from google.colab import userdata

_api_key = userdata.get("LANGCHAIN_API_KEY")

# set ทั้ง family เก่าและใหม่ — กันทุก SDK version
os.environ.update({
    # ชื่อใหม่ (langsmith >= 0.1.x preferred)
    "LANGSMITH_TRACING": "true",
    "LANGSMITH_API_KEY": _api_key,
    "LANGSMITH_ENDPOINT": "https://api.smith.langchain.com",
    "LANGSMITH_PROJECT": "financial-analyst-agent",
    # ชื่อเก่า (backward compat)
    "LANGCHAIN_TRACING_V2": "true",
    "LANGCHAIN_API_KEY": _api_key,
    "LANGCHAIN_ENDPOINT": "https://api.smith.langchain.com",
    "LANGCHAIN_PROJECT": "financial-analyst-agent",
})

# verify ทันที — ต้องได้ True ก่อนไปต่อ
import langsmith.utils as ls_utils
print("Tracing enabled:", ls_utils.tracing_is_enabled())
print("Project:", os.environ["LANGSMITH_PROJECT"])

Tracing enabled: False
Project: financial-analyst-agent


In [7]:
import langsmith.utils as ls_utils

ls_utils.get_env_var.cache_clear()
print("Tracing enabled:", ls_utils.tracing_is_enabled())

Tracing enabled: True


In [8]:
from langsmith import Client, tracing_context
from langchain_core.tracers import LangChainTracer

ls_client = Client(
    api_key=userdata.get("LANGCHAIN_API_KEY"),
    api_url="https://api.smith.langchain.com",
)

# tracer สำหรับ LangGraph agent — ผูก project + client ตรง ๆ
tracer = LangChainTracer(
    project_name="financial-analyst-agent",
    client=ls_client,
)

In [ ]:
# ส่ง api_key และ project ตรงๆ ไม่พึ่ง env var
from langsmith import Client
from google.colab import userdata

ls_client = Client(
    api_key=userdata.get("LANGCHAIN_API_KEY"),
    api_url="https://api.smith.langchain.com",
)

print("✅ Client created")

✅ Client created


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from google.colab import userdata
import yfinance as yf
import numpy as np
from datetime import datetime

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langsmith import Client, traceable
from langsmith.run_helpers import get_current_run_tree

print("✅ All imports OK")

# สร้าง project ถ้ายังไม่มี
try:
    project = ls_client.read_project(project_name="financial-analyst-agent")
    print("✅ Project exists:", project.name)
except Exception:
    project = ls_client.create_project(
        "financial-analyst-agent",
        description="Financial Analyst Agent with ReAct + Hurst Exponent"
    )
    print("✅ Project created:", project.name)

✅ All imports OK
✅ Project exists: financial-analyst-agent


In [ ]:
import requests
key = userdata.get("LANGCHAIN_API_KEY")
r = requests.get(
    "https://api.smith.langchain.com/sessions",
    headers={"x-api-key": key}
)
print(r.status_code)  # 200 = OK, 401 = key ผิด

200


In [13]:
# Cell 1: Setup — เพิ่ม os.environ.clear ก่อน หรือ force overwrite
import os
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from google.colab import userdata
import yfinance as yf
import numpy as np
from datetime import datetime

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langsmith import Client, traceable
from langsmith.run_helpers import get_current_run_tree

# ✅ Force overwrite — ล้างค่าเก่าที่ค้างอยู่
for key in ["LANGCHAIN_TRACING_V2", "LANGCHAIN_API_KEY", "LANGCHAIN_ENDPOINT",
            "LANGCHAIN_PROJECT", "LANGSMITH_PROJECT"]:
    os.environ.pop(key, None)

os.environ.update({
    "LANGCHAIN_TRACING_V2": "true",
    "LANGCHAIN_API_KEY": userdata.get("LANGCHAIN_API_KEY"),
    "LANGCHAIN_ENDPOINT": "https://api.smith.langchain.com",
    "LANGCHAIN_PROJECT": "financial-analyst-agent",
    "GOOGLE_API_KEY": userdata.get("GEMINI_API_KEY"),
})

# ✅ Verify
print("PROJECT:", os.environ.get("LANGCHAIN_PROJECT"))

ls_client = Client()
try:
    project = ls_client.read_project(project_name="financial-analyst-agent")
    print("✅ Connected:", project.name)
except Exception:
    project = ls_client.create_project("financial-analyst-agent")
    print("✅ Created:", project.name)

PROJECT: financial-analyst-agent
✅ Connected: financial-analyst-agent


In [ ]:
# @traceable(name="test_trace", run_type="chain")
# def test_function(x: str) -> str:
#     return f"hello {x}"

# test_function("world")

# import time; time.sleep(2)

# runs = list(ls_client.list_runs(project_name="financial-analyst-agent", limit=3))
# print(f"Runs found: {len(runs)}")

In [10]:
from langsmith import traceable
from langchain_core.tools import tool

# ✅ วิธีที่ถูก: traceable ห่อ logic ข้างใน, @tool อยู่นอกสุด
@tool
def get_stock_price(ticker: str) -> str:
    """Fetch current stock price and key metrics."""
    return _fetch_stock_price_logic(ticker)

@traceable(
    name="fetch_stock_price",
    run_type="tool",
    tags=["market-data", "yfinance"],
)
def _fetch_stock_price_logic(ticker: str) -> str:
    """Internal traceable function."""
    try:
        stock = yf.Ticker(ticker.upper())
        hist = stock.history(period="5d")
        if hist.empty:
            return f"No data for {ticker}"

        latest = hist['Close'].iloc[-1]
        prev = hist['Close'].iloc[-2] if len(hist) > 1 else latest
        change_pct = ((latest - prev) / prev) * 100
        info = stock.info

        return (
            f"Ticker: {ticker.upper()}\n"
            f"Price: ${latest:.2f} (Change: {change_pct:+.2f}%)\n"
            f"52W Range: ${info.get('fiftyTwoWeekLow','N/A')} – ${info.get('fiftyTwoWeekHigh','N/A')}\n"
            f"P/E (TTM): {info.get('trailingPE','N/A')} | Forward P/E: {info.get('forwardPE','N/A')}\n"
            f"Market Cap: ${info.get('marketCap',0)/1e9:.1f}B"
        )
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def get_stock_financials(ticker: str) -> str:
    """Get fundamental financial metrics for analysis."""
    return _fetch_financials_logic(ticker)

@traceable(
    name="fetch_financials",
    run_type="tool",
    tags=["fundamentals", "yfinance"],
)
def _fetch_financials_logic(ticker: str) -> str:
    try:
        info = yf.Ticker(ticker.upper()).info
        return (
            f"Revenue (TTM): ${info.get('totalRevenue',0)/1e9:.1f}B\n"
            f"Net Income: ${info.get('netIncomeToCommon',0)/1e9:.1f}B\n"
            f"Profit Margin: {info.get('profitMargins',0)*100:.1f}%\n"
            f"Revenue Growth YoY: {info.get('revenueGrowth',0)*100:.1f}%\n"
            f"EPS (TTM): ${info.get('trailingEps','N/A')}\n"
            f"Debt/Equity: {info.get('debtToEquity','N/A')}"
        )
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def get_hurst_exponent(ticker: str) -> str:
    """Calculate Hurst exponent to detect market regime."""
    return _calc_hurst_logic(ticker)

@traceable(
    name="calc_hurst_exponent",
    run_type="tool",
    tags=["quant", "regime-detection"],
)
def _calc_hurst_logic(ticker: str) -> str:
    try:
        hist = yf.Ticker(ticker.upper()).history(period="1y")["Close"]
        returns = np.log(hist / hist.shift(1)).dropna().values

        lags = range(2, 20)
        rs_values = []
        for lag in lags:
            segments = [returns[i:i+lag] for i in range(0, len(returns)-lag, lag)]
            rs_list = [
                (np.max(np.cumsum(s - np.mean(s))) - np.min(np.cumsum(s - np.mean(s)))) / np.std(s)
                for s in segments if np.std(s) > 0
            ]
            if rs_list:
                rs_values.append(np.mean(rs_list))

        hurst = np.polyfit(np.log(list(lags)[:len(rs_values)]), np.log(rs_values), 1)[0]

        if hurst > 0.55:
            regime = "📈 Trending — momentum strategies work"
        elif hurst < 0.45:
            regime = "↔️ Mean-Reverting — RSI/Bollinger strategies work"
        else:
            regime = "🎲 Random Walk — harder to predict"

        return f"Hurst Exponent ({ticker.upper()}, 1Y): {hurst:.4f}\nRegime: {regime}"
    except Exception as e:
        return f"Error: {str(e)}"

In [11]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    reasoning_effort="low",   # พอสำหรับ tool orchestration, ประหยัด tokens
    api_key=userdata.get("GROQ_API_KEY"),
)

In [19]:
from langsmith.run_helpers import get_current_run_tree


In [22]:
# Cell 3: Agent + Tracing wrapper

# model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.2)
tools = [get_stock_price, get_stock_financials, get_hurst_exponent]

SYSTEM_PROMPT = """You are a quantitative financial analyst assistant.
Always fetch real-time data before answering.
Provide objective analysis with data. Note that this is not financial advice.
Respond in Thai mixed with English technical terms."""

agent_graph = create_react_agent(model, tools, prompt=SYSTEM_PROMPT)


@traceable(
    name="financial_analyst_agent",
    run_type="chain",
    tags=["agent", "financial-analysis"],
    project_name="financial-analyst-agent",
    client=ls_client,
)
def run_financial_agent(
    query: str,
    tickers: list[str] = None,
    analysis_type: str = "general",
) -> dict:
    """
    Main entry point — wrapped ด้วย @traceable เพื่อ group ทุก sub-runs ไว้ใน 1 parent trace
    """
    config = RunnableConfig(
        run_name=f"query_{analysis_type}_{datetime.now().strftime('%H%M%S')}",
        callbacks=[tracer],
        tags=[analysis_type] + (tickers or []),
        metadata={
            "query": query,
            "tickers": tickers or [],
            "analysis_type": analysis_type,
            "timestamp": datetime.now().isoformat(),
        },
    )

    inputs = {"messages": [HumanMessage(content=query)]}
    final_response = ""

    print(f"\n{'='*55}")
    print(f"🔍 Query: {query[:80]}...")
    print(f"{'='*55}")

    for event in agent_graph.stream(inputs, config=config, stream_mode="values"):
        if "messages" in event:
            last = event["messages"][-1]
            last.pretty_print()
            if hasattr(last, "content") and last.content:
                final_response = last.content

    rt = get_current_run_tree()
    return {
        "query": query,
        "response": final_response,
        "tickers": tickers,
        "analysis_type": analysis_type,
        "run_id": str(rt.id) if rt else None,   # ได้ ID ของ trace นี้แน่นอน
    }

In [23]:
# Cell 4: Run + ดู Trace URL

with tracing_context(enabled=True, client=ls_client):
    result = run_financial_agent(
        query="วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)",
        tickers=["NVDA"],
        analysis_type="full_analysis",
    )

ls_client.flush()

# ดึง trace URL มาแสดง
import time
time.sleep(5)   # เผื่อ server-side ingest

runs = list(ls_client.list_runs(
    project_name="financial-analyst-agent",
    is_root=True,
    limit=1,
))

if runs:
    print(f"\n🔗 LangSmith Trace URL:")
    print(runs[0].url)          # 👈 private URL เปิดดูเองได้เลย
else:
    print("⚠️ ยังไม่เจอ run — รอเพิ่มหรือเช็ค web UI")


🔍 Query: วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)...
================================ Human Message =================================

วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (fc_7d38feff-e2b7-4546-828f-c24afc8e745d)
 Call ID: fc_7d38feff-e2b7-4546-828f-c24afc8e745d
  Args:
    ticker: NVDA
================================= Tool Message =================================
Name: get_stock_price

Ticker: NVDA
Price: $nan (Change: +nan%)
52W Range: $140.85 – $236.54
P/E (TTM): 31.863276 | Forward P/E: 15.7474375
Market Cap: $4854.4B
================================== Ai Message ==================================
Tool Calls:
  get_stock_financials (fc_d42eeafc-e6bc-427a-9bd8-bb4438e8fee2)
 Call ID: fc_d42eeafc-e6bc-427a-9bd8-bb4438e8fee2
  Args:
    ticker: NVDA
================================= Tool Messa

In [24]:
# วิธีที่ 1: ได้ ID เป๊ะ ๆ จากข้างใน function (แม่นสุด)
run_id = result["run_id"]

# private URL — เปิดดูเองใน workspace ตัวเอง
import time; time.sleep(5)
run = ls_client.read_run(run_id)
print("🔒 Private URL:", run.url)

# public URL — ต้อง share ก่อนถึงจะได้ลิงก์ /public/
shared_url = ls_client.share_run(run_id)
print("🌐 Public URL:", shared_url)   # อันนี้แหละคือ /public/{token}/r ตัวจริง

🔒 Private URL: https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019eb540-1d70-7e53-8c05-26e9591406a1?trace_id=019eb540-1d70-7e53-8c05-26e9591406a1&start_time=2026-06-11T05:55:38.480082
🌐 Public URL: https://smith.langchain.com/public/caca6e8b-3238-4700-9c2f-16a37ad0733e/r


In [25]:
# วิธีที่ 2: is_root=True — ใช้เป็น sanity check / ดูภาพรวม
runs = list(ls_client.list_runs(
    project_name="financial-analyst-agent",
    is_root=True,
    limit=5,
))
for r in runs:
    print(r.start_time, r.name, r.url)

2026-06-11 05:55:38.480082+00:00 financial_analyst_agent https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019eb540-1d70-7e53-8c05-26e9591406a1?trace_id=019eb540-1d70-7e53-8c05-26e9591406a1&start_time=2026-06-11T05:55:38.480082
2026-06-11 05:53:26.424345+00:00 financial_analyst_agent https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019eb53e-1998-7280-8cfa-461b660900bd?trace_id=019eb53e-1998-7280-8cfa-461b660900bd&start_time=2026-06-11T05:53:26.424345
2026-06-11 05:31:05.215880+00:00 financial_analyst_agent https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019eb529-a27f-72b2-a251-e776fa9be0da?trace_id=019eb529-a27f-72b2-a251-e776fa9be0da&start_time=2026-06-11T05:31:05.215880
2026-06-11 04:31:42.429333+00:00 financial_analyst_agent https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e224